# SHIPP Stage 7B — Silver routes

`ors_route_responses` → `silver_routes`. **Grain:** one row per `(origin_key, dest_key, profile)`,
from the latest successful response. `route_status` = `OK` or `NO_ROUTE`; distance is never invented.
Logic: `data_pipeline/silver/routes.py` (unit-tested). Rebuilt from Bronze each run → idempotent.

**Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert. Only writer of `silver_routes`.

In [0]:
%run ../common/shipp_silver_lib

In [0]:
BRONZE_ROUTE_RESPONSES = f"{CATALOG}.{BRONZE_SCHEMA}.ors_route_responses"

In [0]:
import config.tables as t

print("Loaded config file:", t.__file__)
print("SILVER_CANDIDATE_PAIRS:", t.SILVER_CANDIDATE_PAIRS)
print("BRONZE_ROUTE_RESPONSES:", t.BRONZE_ROUTE_RESPONSES)
print("SILVER_ROUTES:", t.SILVER_ROUTES)
print("PIPELINE_RUN_LOG:", t.PIPELINE_RUN_LOG)

print("PASS — centralized table registry loaded")

In [0]:
from data_pipeline.silver.routes import ROUTE_COLUMNS, parse_route_responses

assert table_exists(BRONZE_ROUTE_RESPONSES), "No Bronze route responses yet — run 31 first."
EXPECTED_COLUMNS = ROUTE_COLUMNS + ["silver_processed_at"]

routes_df = parse_route_responses(spark.table(BRONZE_ROUTE_RESPONSES)) \
    .withColumn("silver_processed_at", F.current_timestamp())
row_count = routes_df.count()
display(routes_df.orderBy("dest_key", "distance_km"))

In [0]:
uncovered = (
    spark.table(SILVER_CANDIDATE_PAIRS)
    .select(
        "origin_key",
        "dest_key",
        "profile",
    )
    .distinct()
    .join(
        spark.table(SILVER_ROUTES),
        [
            "origin_key",
            "dest_key",
            "profile",
        ],
        "left_anti",
    )
    .count()
)

print(
    "Pair route keys still without a route:",
    uncovered,
)

assert uncovered == 0, (
    f"FAILED: {uncovered} candidate route keys "
    "do not have a Silver route."
)

print("PASS — Stage 32 complete.")

In [0]:
from data_pipeline.silver.routes import (
    ROUTE_COLUMNS,
    parse_route_responses,
)

assert table_exists(BRONZE_ROUTE_RESPONSES), (
    "No Bronze route responses yet — run 31 first."
)

EXPECTED_COLUMNS = ROUTE_COLUMNS + [
    "silver_processed_at"
]

routes_df = (
    parse_route_responses(
        spark.table(BRONZE_ROUTE_RESPONSES)
    )
    .withColumn(
        "silver_processed_at",
        F.current_timestamp(),
    )
)

row_count = routes_df.count()

display(
    routes_df.orderBy(
        "dest_key",
        "distance_km",
    )
)

run_quality_gate(
    {
        "duplicate route key": duplicate_key_count(
            routes_df,
            "origin_key",
            "dest_key",
            "profile",
        ),
        "negative distance": routes_df.filter(
            F.col("distance_km") < 0
        ).count(),
        "negative duration": routes_df.filter(
            F.col("duration_min") < 0
        ).count(),
        "OK without distance": routes_df.filter(
            (F.col("route_status") == "OK")
            & F.col("distance_km").isNull()
        ).count(),
        "NO_ROUTE with distance": routes_df.filter(
            (F.col("route_status") == "NO_ROUTE")
            & F.col("distance_km").isNotNull()
        ).count(),
    },
    row_count=row_count,
    allow_empty=True,
)

written = write_overwrite(
    routes_df,
    SILVER_ROUTES,
)

assert written == row_count, (
    f"WRITE VALIDATION FAILED: "
    f"wrote {written}, expected {row_count}"
)

assert_exact_schema(
    SILVER_ROUTES,
    EXPECTED_COLUMNS,
)

uncovered = (
    spark.table(SILVER_CANDIDATE_PAIRS)
    .select(
        "origin_key",
        "dest_key",
        "profile",
    )
    .distinct()
    .join(
        spark.table(SILVER_ROUTES),
        [
            "origin_key",
            "dest_key",
            "profile",
        ],
        "left_anti",
    )
    .count()
)

print(
    "Pair route keys still without a route "
    "(retried next run):",
    uncovered,
)

summary = {
    "table": SILVER_ROUTES,
    "rows": written,
    "uncovered_pairs": uncovered,
    "status": "PASS",
}

print(summary)

dbutils.notebook.exit(
    json.dumps(summary)
)

In [0]:
run_quality_gate({
    "duplicate route key": duplicate_key_count(routes_df, "origin_key", "dest_key", "profile"),
    "negative distance": routes_df.filter(F.col("distance_km") < 0).count(),
    "negative duration": routes_df.filter(F.col("duration_min") < 0).count(),
    "OK without distance": routes_df.filter((F.col("route_status") == "OK") & F.col("distance_km").isNull()).count(),
    "NO_ROUTE with distance": routes_df.filter((F.col("route_status") == "NO_ROUTE") & F.col("distance_km").isNotNull()).count(),
}, row_count=row_count, allow_empty=True)

written = write_overwrite(routes_df, SILVER_ROUTES)
assert written == row_count, f"WRITE VALIDATION FAILED: wrote {written}, expected {row_count}"
assert_exact_schema(SILVER_ROUTES, EXPECTED_COLUMNS)

uncovered = (spark.table(SILVER_CANDIDATE_PAIRS).select("origin_key", "dest_key", "profile").distinct()
             .join(spark.table(SILVER_ROUTES), ["origin_key", "dest_key", "profile"], "left_anti").count())
print("Pair route keys still without a route (retried next run):", uncovered)

summary = {"table": SILVER_ROUTES, "rows": written, "uncovered_pairs": uncovered, "status": "PASS"}
print(summary)
dbutils.notebook.exit(json.dumps(summary))